# AutoDQ sales workflow — Jupyter version

This notebook mirrors `examples/sales_analysis.adql` using AutoDQ's native Python API. It profiles and cleans the sales dataset, engineers features, builds reusable visualizations, trains a revenue model, estimates prediction uncertainty, produces SHAP and BLUE diagnostics, and exports the final artifacts.

Run the cells from top to bottom. The `project` object retains the workflow state between cells.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from autodq import AutoDQ

DATASET = Path("../datasets/sample/sales.csv")
REPORTS = Path("../reports")
EXPORTS = Path("../exports")
MODELS = Path("../models")

for directory in (REPORTS, EXPORTS, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
project = AutoDQ(str(DATASET))
data = project.load()
project.set_target("Revenue")

print(f"Loaded {DATASET.name}: {len(project.state.data):,} rows × {len(project.state.data.columns)} columns")
display(project.head())

## 1. Data quality

Generate the profile, descriptive statistics, interpretations, diagnosis, and evidence-aware recommendations.

In [ ]:
profile_report = project.profile()
project.show_profile()

statistics_report = project.statistics()
project.show_statistics()

interpretation_report = project.interpret()
project.show_interpretations()

diagnosis_report = project.diagnose()
project.show_diagnosis()

recommendations = project.recommend()
project.show_recommendations()

## 2. Interactive cleaning and domain review

Build the proposed cleaning plan, inspect its previews, apply business-domain rules, and review row-level outliers before approving any changes.

In [ ]:
knowledge_rules = project.apply_knowledge()
project.show_knowledge()

decision_plan = project.decide()
preview_report = project.preview()
project.show_preview()

review = project.review_cleaning(refresh=True)
review

In [ ]:
domain_rules = [
    project.add_domain_rule(
        "Revenue",
        min_value=0,
        nullable=False,
        description="Revenue must be present and non-negative",
    ),
    project.add_domain_rule(
        "Customer_Age",
        min_value=18,
        max_value=100,
        description="Expected customer age range",
    ),
    project.add_domain_rule(
        "Customer_Satisfaction",
        min_value=1,
        max_value=5,
        description="Satisfaction uses a five-point scale",
    ),
    project.add_domain_rule(
        "Region",
        allowed_values=["North", "South", "East", "West", "Central"],
    ),
]

display(pd.DataFrame([rule.to_dict() for rule in domain_rules]))

domain_before = project.validate_domain()
display(domain_before)

outlier_report = project.review_outliers(
    columns=["Revenue", "Profit", "Customer_Age"],
    iqr_multiplier=1.5,
)
display(outlier_report)

## 3. Apply approved cleaning

Approve the generated actions, cap reviewed Revenue extremes at the IQR bounds, preview the effects, clean the working copy, and validate the result. The source CSV is never overwritten.

In [ ]:
review = project.approve_all()
changed_outliers = project.treat_outliers(
    column="Revenue",
    strategy="clip",
    iqr_multiplier=1.5,
    reason="Cap reviewed revenue extremes at the IQR bounds",
)

print(f"Revenue outlier rows treated: {changed_outliers:,}")
display(project.cleaning_preview(max_rows=3))

cleaned_data = project.clean()
project.show_cleaning_report()

validation_report = project.validate_cleaning()
project.show_validation()

domain_after = project.validate_domain()
display(domain_after)

## 4. Feature engineering

Evaluate relationships and ML readiness, apply AutoDQ's recommended features, and add three transparent business features that do not directly use the Revenue target.

In [ ]:
correlation_report = project.correlation(min_abs_correlation=0.25)
project.show_correlation()

readiness_report = project.ml_readiness()
project.show_ml_readiness()

feature_report = project.features()
project.show_features()
engineered_data = project.apply_features()

engineered_data = project.create_feature(
    name="Customer_Age_Squared",
    method="square",
    column="Customer_Age",
)
engineered_data = project.create_feature(
    name="Delivery_Is_Slow",
    method="flag_greater_equal",
    column="Delivery_Days",
    expression="5",
)
engineered_data = project.create_feature(
    name="Order_Year",
    method="date_year",
    column="Date",
)

display(engineered_data[["Customer_Age_Squared", "Delivery_Is_Slow", "Order_Year"]].head())

## 5. Regional and product analysis

These pandas aggregations are the direct notebook equivalents of the two ADQL `SELECT` cells.

In [ ]:
regional_analysis = (
    cleaned_data.dropna(subset=["Revenue", "Region"])
    .groupby("Region", as_index=False)
    .agg(
        total_revenue=("Revenue", "sum"),
        average_profit=("Profit", "mean"),
        average_satisfaction=("Customer_Satisfaction", "mean"),
        transactions=("Transaction_ID", "size"),
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
)
display(regional_analysis)

In [ ]:
product_analysis = (
    cleaned_data.dropna(subset=["Product_Category"])
    .groupby("Product_Category", as_index=False)
    .agg(
        total_revenue=("Revenue", "sum"),
        total_profit=("Profit", "sum"),
        average_discount=("Discount_Rate", "mean"),
        transactions=("Transaction_ID", "size"),
    )
    .sort_values("total_revenue", ascending=False)
    .head(15)
)
display(product_analysis)

## 6. Reusable visualizations and gallery

Generate the same four customized charts as ADQL, retain them in the project gallery, customize the regional chart, and save the complete gallery.

In [ ]:
regional_chart_report = project.visualize(
    chart="bar",
    x="Region",
    y="Revenue",
    stage="cleaned",
    title="Revenue by Region",
    subtitle="Average cleaned revenue by sales region",
    x_label="Region",
    y_label="Average revenue",
    theme="light",
    palette="cool",
    figsize=(10, 6),
    dpi=160,
    grid=True,
    legend=False,
)

revenue_histogram_report = project.visualize(
    chart="histogram",
    column="Revenue",
    stage="cleaned",
    title="Revenue Distribution",
    x_label="Revenue",
    y_label="Transactions",
    theme="journal",
    color="#2563eb",
    figsize=(10, 6),
    dpi=160,
    grid=True,
)

revenue_profit_report = project.visualize(
    chart="scatter",
    x="Revenue",
    y="Profit",
    stage="engineered",
    title="Revenue and Profit",
    x_label="Revenue",
    y_label="Profit",
    theme="presentation",
    color="#059669",
    figsize=(10, 6),
    dpi=160,
    grid=True,
)

correlation_chart_report = project.visualize(
    chart="correlation_heatmap",
    stage="engineered",
    title="Engineered Feature Correlations",
    theme="light",
    figsize=(12, 9),
    dpi=160,
)

In [ ]:
gallery = project.list_visualizations()
regional_chart = project.customize_visualization(
    "bar_Region_by_Revenue_cleaned",
    subtitle="Cleaned sales data",
    theme="journal",
    dpi=200,
)
saved_charts = project.save_visualizations(
    str(REPORTS / "jupyter-sales-charts"),
    format="png",
)

print(f"Saved {len(saved_charts)} gallery charts.")
project.show_visualizations()

## 7. Revenue model and prediction uncertainty

Train the same leakage-aware random-forest regressor, generate 90% prediction intervals, persist the model, and review the largest predictions.

In [ ]:
model_report = project.model(
    algorithm="random_forest_regressor",
    use_engineered=True,
    exclude_leakage=True,
    test_size=0.2,
    random_state=42,
)
project.show_model()

predictions = project.predict(
    uncertainty=True,
    confidence_level=0.90,
    low_confidence_threshold=0.60,
)
project.show_predictions()

model_bundle = project.save_model(
    str(MODELS / "jupyter-sales-revenue-model"),
    overwrite=True,
)
print(f"Model saved to {model_bundle.path}")

In [ ]:
prediction_review = (
    predictions[[
        "Transaction_ID",
        "Region",
        "Revenue",
        "AutoDQ_Prediction",
        "AutoDQ_Prediction_Lower",
        "AutoDQ_Prediction_Upper",
    ]]
    .sort_values("AutoDQ_Prediction", ascending=False)
    .head(25)
)
display(prediction_review)

## 8. SHAP explainability

Explain a sample of predictions and save the same summary and waterfall plots generated by ADQL.

In [ ]:
explainability_report = project.explain(
    max_rows=20,
    use_engineered=True,
)
project.show_explanations()

shap_summary = project.visualize_shap(
    "summary",
    save=str(REPORTS / "jupyter-sales-shap-summary.png"),
)
shap_waterfall = project.visualize_shap(
    "waterfall",
    row=0,
    save=str(REPORTS / "jupyter-sales-shap-waterfall.png"),
)

## 9. BLUE regression diagnostics

Evaluate linear-model assumptions, render the diagnostic plots, interpret them, and generate prescriptions.

In [ ]:
blue_report = project.blue(
    source="data",
    use_engineered=True,
    exclude_leakage=True,
    max_features=12,
    significance_level=0.05,
)
project.show_blue()

blue_visuals = project.visualize_blue(display=True, append=True)
blue_insights = project.interpret_blue_visuals()
project.show_blue_visual_insights()

blue_prescriptions = project.prescribe_blue()
project.show_blue_prescriptions()

## 10. Dashboard and exports

Build the notebook-ready executive dashboard, then export the report, datasets, predictions, model artifacts, and cleaning audit trail.

In [ ]:
dashboard = project.dashboard(
    output=str(REPORTS / "jupyter-sales-dashboard.html"),
    title="AutoDQ Sales Analytics",
    subtitle="Data quality, cleaning, revenue modeling, uncertainty, and diagnostics",
    theme="executive",
    stage="engineered",
    max_charts=12,
    max_preview_rows=25,
    overwrite=True,
    auto_display=True,
)
dashboard

In [ ]:
project.generate_report(
    str(REPORTS / "jupyter-sales-report.html"),
    style="executive",
)
project.export_cleaned(str(EXPORTS / "jupyter-sales-cleaned.csv"))
project.export_engineered(str(EXPORTS / "jupyter-sales-engineered.csv"))
project.export_predictions(str(EXPORTS / "jupyter-sales-predictions.csv"))
audit_path = project.export_cleaning_audit(
    str(REPORTS / "jupyter-sales-cleaning-audit.json")
)

comparison_summary = pd.DataFrame(
    {
        "result": [
            "Rows loaded",
            "Rows cleaned",
            "Engineered columns",
            "Gallery charts",
            "Predictions",
            "Cleaning audit entries",
        ],
        "value": [
            len(project.state.data),
            len(cleaned_data),
            len(engineered_data.columns),
            len(project.visualization_gallery.charts),
            len(predictions),
            review.audit_count,
        ],
    }
)
display(comparison_summary)
print(f"Cleaning audit exported to {audit_path}")

## Workflow complete

Open the generated dashboard in `reports`, inspect the saved charts and SHAP plots, and compare the `jupyter-sales-*` artifacts with their `adql-sales-*` counterparts.